# HE-IFD -- Experiments (one-shot federated fine-tuning, **no alignment phase**)\n\n**Run All** (Runtime -> Run all). Nothing to configure. Each stage runs unattended and prints its\n`results.csv` inline between `===== BEGIN results.csv =====` markers -- copy it out (no download).\nAll stages share one case (`ft_experiments`) and **resume** each other, so if Colab drops, Run All again.\n\n- **1. Main grid**: headline accuracy + heterogeneity sweep + the necessity check (no_phase0 vs raw_union).\n- **2. Ablations**: client scaling, trajectory length, trainable unit, lambda, distillation.\n- **3. Cost**: the measured CKKS communication/computation (reused, not re-run).\n- **4. Membership inference**: pointer to the separate MIA notebook.\n\nMetric columns in the CSV: `acc` (global model), `theta0_acc` (the fixed init alone), `m4_ood_acc`\n(coverage of classes a client never saw), and the teacher/oracle references.

In [ ]:
# SETUP (no input). Clone, deps, prefetch, online.
import os, sys, subprocess
REPO_DIR, REPO_URL = "/content/HE-IFD", "https://github.com/hkanpak21/HE-IFD.git"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone","-q",REPO_URL,REPO_DIR], check=False)
os.chdir(REPO_DIR)
subprocess.run(["git","fetch","-q","origin","master"], check=False)
subprocess.run(["git","checkout","-q","origin/master","--","src","jobs","tests"], check=False)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

# Pin datasets<3.0 -- newer datasets removed the script loaders Banking77/TREC/DBpedia still ship.
subprocess.run([sys.executable,"-m","pip","-q","install",
                "datasets<3.0","transformers","timm","ipywidgets"], check=False)
os.environ["HF_DATASETS_TRUST_REMOTE_CODE"] = "1"

import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

# Vision: download directly via torchvision (no datasets lib).
import torchvision as tv
for ds in (tv.datasets.CIFAR100, tv.datasets.CIFAR10):
    ds("data", train=True, download=True); ds("data", train=False, download=True)
print("cifar100 / cifar10 ready")

# Text datasets + the fine-grained vision dataset, online & idempotent. Split so one failure can't cascade.
for flags in (["--include-text019"], ["--include-ft02-text"], ["--include-ft02-fgvc"]):
    print("prefetching:", flags)
    subprocess.run([sys.executable,"jobs/prefetch_login.py","--data-root","data",*flags], check=False)

os.environ.pop("HF_HUB_OFFLINE", None); os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("setup done -- online; datasets cached.")

## 1. Main grid

In [ ]:
# 1. MAIN GRID -- headline accuracy, heterogeneity sweep, and the necessity check in one grid.
# no_phase0 is THE method (fixed public init, NO alignment). raw_union is the comparison: if it matches
# no_phase0, the alignment phase is unnecessary. LoRA + direct fine-tuning. 3 seeds. ~120 cells.
from src.notebook_runner import run_unattended
run_unattended(dict(
    backbones=["roberta_base_banking77", "roberta_base_dbpedia", "roberta_base_trec",
               "vit_b32_cifar100", "vit_b32_fgvc_aircraft"],
    Ns=[10], alphas=[0.05, 0.1, 0.3, 1.0],
    methods=["no_phase0", "raw_union_K300"],
    seeds=[42, 43, 44], Ks=[300],
    trainable_units=["lora"], local_steps=["finetune"],
    case="ft_experiments",
))

## 2. Ablations

In [ ]:
# 2. ABLATIONS -- scaling in N, trajectory length K, trainable unit, lambda, distillation.
# All on no_phase0 (the method), Banking77, at alpha 0.05 (skew) and 1.0 (mild). Same case -> resumes.
from src.notebook_runner import run_unattended

# client scaling
run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[5, 20, 50], alphas=[0.05, 1.0],
    methods=["no_phase0"], seeds=[42, 43, 44], Ks=[300],
    trainable_units=["lora"], local_steps=["finetune"], case="ft_experiments"))

# trajectory length K (bounded fine-tuning steps)
run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[10], alphas=[0.05, 1.0],
    methods=["no_phase0"], seeds=[42, 43, 44], Ks=[50, 100, 300, 600],
    trainable_units=["lora"], local_steps=["finetune"], case="ft_experiments"))

# trainable unit: head (linear probe) vs LoRA vs last-N blocks
run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[10], alphas=[0.05, 1.0],
    methods=["no_phase0"], seeds=[42, 43, 44], Ks=[300],
    trainable_units=["head", "lora", "last_n"], local_steps=["finetune"], case="ft_experiments"))

# lambda scaling-coefficient curve (eval-only drift regularizer)
run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[10], alphas=[0.05, 1.0],
    methods=["no_phase0"], seeds=[42], Ks=[300], trainable_units=["lora"], local_steps=["finetune"],
    lambda_scales=[0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0], case="ft_experiments"))

# distillation ablation (teacher-based local step vs direct fine-tuning)
run_unattended(dict(backbones=["roberta_base_banking77"], Ns=[10], alphas=[0.05, 1.0],
    methods=["no_phase0"], seeds=[42, 43, 44], Ks=[300],
    trainable_units=["lora"], local_steps=["distill"], case="ft_experiments"))

## 3. Encrypted-aggregation cost (already measured -- reused here)

Measured end to end on a real multiparty CKKS implementation (Lattigo, ring $2^{14}$, depth 1; Apple M4).
No need to re-run; the cost is set by the trainable-parameter count, not the backbone.

| trainable unit | ct/client | up=down (N=10 / 100) | DKG (N=10/100) | encrypt/client | aggregate (N=10/100) | decrypt (N=10/100) | rel. L2 |
|---|---|---|---|---|---|---|---|
| 4-class head (3.1k) | 1 | 5.0 / 50 MiB | 13 / 96 ms | ~2.3 ms | 3.9 / 38.9 ms | 2.4 / 23.2 ms | $10^{-9}$..$10^{-8}$ |
| 100-class head (76.9k) | 10 | 50 / 500 MiB | 11 / 98 ms | ~21.5 ms | 39.0 / 382 ms | 24.6 / 217 ms | $10^{-9}$..$10^{-8}$ |

A LoRA adapter adds $2dr$ parameters per adapted layer (a few thousand) -> a small, fixed number of extra
ciphertexts, depth still 1. The Go proof-of-concept lives in `fhe/`.


## 4. Membership inference / residual leakage

Run separately via **`notebooks/colab_028_mia.ipynb`** (64 shadow models per cell; heavier). It attacks the
released `no_phase0` models with Yeom / LiRA / GLiRA. Because there is no alignment phase, the only exposed
surface is the released model itself -- there is no prototype channel to attack.
